In [2]:
!pip install -q qiskit qiskit-aer

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.8/9.8 MB 68.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 89.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 104.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.9/54.9 kB 5.4 MB/s eta 0:00:00


**Easy**

In [3]:
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator

qc = QuantumCircuit(2, 2)

# Create shared Bell pair
qc.h(0)
qc.cx(0, 1)

# Alice encodes message 00
# No gate required

# Bob decodes
qc.cx(0, 1)
qc.h(0)

qc.measure([0, 1], [0, 1])

print(qc.draw())

simulator = AerSimulator()
result = simulator.run(qc, shots=1024).result()

print("Decoded message:", result.get_counts())

     ┌───┐          ┌───┐┌─┐
q_0: ┤ H ├──■────■──┤ H ├┤M├
     └───┘┌─┴─┐┌─┴─┐└┬─┬┘└╥┘
q_1: ─────┤ X ├┤ X ├─┤M├──╫─
          └───┘└───┘ └╥┘  ║ 
c: 2/═════════════════╩═══╩═
                      1   0 
Decoded message: {'00': 1024}


**Medium**

In [4]:
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator

def superdense_coding(message):
    qc = QuantumCircuit(2, 2)

    # Create Bell pair
    qc.h(0)
    qc.cx(0, 1)

    # Alice encodes the message
    if message[1] == '1':
        qc.x(0)
    if message[0] == '1':
        qc.z(0)

    # Bob decodes
    qc.cx(0, 1)
    qc.h(0)

    qc.measure([0, 1], [0, 1])
    return qc

simulator = AerSimulator()

for message in ['00', '01', '10', '11']:
    qc = superdense_coding(message)
    result = simulator.run(qc, shots=1024).result()
    print(message, "->", result.get_counts())

00 -> {'00': 1024}
01 -> {'10': 1024}
10 -> {'01': 1024}
11 -> {'11': 1024}


**Hard**

In [5]:
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator

qc = QuantumCircuit(2, 2)

# Create Bell pair
qc.h(0)
qc.cx(0, 1)

# Alice encodes message 00
# No encoding gate required

# Simulate a bit-flip error on transmitted qubit
qc.x(0)

# Bob decodes
qc.cx(0, 1)
qc.h(0)

qc.measure([0, 1], [0, 1])

result = AerSimulator().run(qc, shots=1024).result()
print("Output with bit-flip error:", result.get_counts())

Output with bit-flip error: {'10': 1024}


**Real-World**

In [6]:
messages = 100

# Classical communication
bits_per_message = 2
classical_bits = messages * bits_per_message

# Superdense coding
qubits_per_message = 1
quantum_bits = messages * qubits_per_message

print("Number of messages:", messages)
print("Classical bits required:", classical_bits)
print("Qubits transmitted using superdense coding:", quantum_bits)
print("Classical bits per message:", bits_per_message)
print("Classical bits encoded per transmitted qubit:", 2)

Number of messages: 100
Classical bits required: 200
Qubits transmitted using superdense coding: 100
Classical bits per message: 2
Classical bits encoded per transmitted qubit: 2


**Challenge**

In [7]:
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel, pauli_error

# Set bit-flip probability
p = 0.1

# Apply X error with probability p
noise_model = NoiseModel()
error = pauli_error([('X', p), ('I', 1 - p)])
noise_model.add_all_qubit_quantum_error(error, ['id'])

qc = QuantumCircuit(2, 2)

# Create Bell pair
qc.h(0)
qc.cx(0, 1)

# Encode message 00
qc.id(0)  # Noisy channel acts here

# Decode
qc.cx(0, 1)
qc.h(0)

qc.measure([0, 1], [0, 1])

simulator = AerSimulator(noise_model=noise_model)
result = simulator.run(qc, shots=10000).result()
counts = result.get_counts()

print("Measurement results:", counts)

correct = counts.get('00', 0)
error_rate = (10000 - correct) / 10000 * 100

print("Decoding error rate:", round(error_rate, 2), "%")

Measurement results: {'10': 1018, '00': 8982}
Decoding error rate: 10.18 %
